In [ ]:
# deploy_endpoint.ipynb

# 1. Setup
import boto3
import sagemaker
import tarfile
import os

# Gunakan role LabRole yang sudah ada di Learner Lab
role = 'LabRole'  # Biasanya langsung bisa digunakan, atau lengkapi ARN: arn:aws:iam::<account-id>:role/LabRole
bucket = 'technomart-s3-test'  # Ganti dengan nama bucket Anda
sagemaker_session = sagemaker.Session()

print(f"Bucket: {bucket}")
print(f"Region: {sagemaker_session.boto_region_name}")

# 2. Download model dan matrix dari S3 ke lokal
s3 = boto3.client('s3')

s3.download_file(bucket, 'models/recommender_model.pkl', 'model.pkl')
s3.download_file(bucket, 'models/interaction_matrix.npz', 'interaction_matrix.npz')
print("Model dan matrix berhasil didownload.")

# 3. Buat tarball (model.tar.gz) yang berisi kedua file
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add('model.pkl')
    tar.add('interaction_matrix.npz')
print("Tarball model.tar.gz berhasil dibuat.")

# 4. Upload tarball ke S3
s3.upload_file('model.tar.gz', bucket, 'models/recommender_model.tar.gz')
print("Tarball diupload ke S3.")

# 5. Deploy ke endpoint SageMaker
from sagemaker.sklearn.model import SKLearnModel

model_data = f's3://{bucket}/models/recommender_model.tar.gz'

# Buat objek model
model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',      # file inference.py harus ada di direktori yang sama
    source_dir='.',                   # direktori saat ini (tempat inference.py & requirements.txt)
    dependencies=['requirements.txt'],
    framework_version='1.2-1',        # container scikit-learn (sudah include numpy/scipy)
    sagemaker_session=sagemaker_session
)

# Deploy endpoint (pilih instance type sesuai kebutuhan)
predictor = model.deploy(
    instance_count=1,
    instance_type='ml.m5.large',      # bisa diganti dengan ml.t2.medium untuk uji coba ringan
    endpoint_name='recommender-endpoint'
)

print(f"Endpoint {predictor.endpoint_name} berhasil dibuat. Tunggu hingga status InService.")